# Biblical Cross-Encoder Fine-Tuning

This notebook fine-tunes a Cross-Encoder model specifically for "Pastor Paraphrase" matching. 

**Goal:** Transform colloquial phrases like *"Peter sinking in the waves"* into a precision match for *Matthew 14:30*.

**Runtime Requirement:** GPU (T4 or better).

In [ ]:
# 1. Install dependencies
!pip install -q sentence-transformers optimum[onnxruntime] torch pandas sqlite3

In [ ]:
# 2. Download Bible Data
import urllib.request
import sqlite3
import pandas as pd

DB_URL = 'https://raw.githubusercontent.com/alshival/super_bible/main/SUPER_BIBLE/super_bible.db'
urllib.request.urlretrieve(DB_URL, 'super_bible.db')

conn = sqlite3.connect('super_bible.db')
df = pd.read_sql_query("SELECT title, chapter, verse, text FROM super_bible WHERE language='EN' AND version='KJV'", conn)
conn.close()

print(f"Loaded {len(df)} verses for training.")

In [ ]:
# 3. Synthetic Data Generation (The "Biblical Schema")
from sentence_transformers import InputExample
import random

train_samples = []

def create_paraphrases(verse_text):
    # This simulates common ways pastors speak:
    clean = verse_text.replace("thee", "you").replace("thou", "you").replace("shall", "will").replace("unto", "to")
    return [
        f"that part where it says {clean[:50]}...",
        f"{clean.lower()}",
        f"the verse about {clean.split()[-1]} and {clean.split()[0]}"
    ]

target_verses = df.sample(2000).to_dict('records')

for row in target_verses:
    verse_text = row['text']
    ref = f"{row['title']} {row['chapter']}:{row['verse']}"
    paras = create_paraphrases(verse_text)
    for p in paras:
        train_samples.append(InputExample(texts=[p, verse_text], label=1.0))
        train_samples.append(InputExample(texts=[p, ref + " " + verse_text], label=1.0))
    wrong_verse = df[df['title'] == row['title']].sample(1).iloc[0]['text']
    if wrong_verse != verse_text:
        train_samples.append(InputExample(texts=[paras[0], wrong_verse], label=0.0))

print(f"Generated {len(train_samples)} training pairs.")

In [ ]:
# 4. Training
from sentence_transformers import CrossEncoder
from torch.utils.data import DataLoader

model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', num_labels=1, device='cuda')

train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=16)

print("Starting Fine-tuning...")
model.fit(train_dataloader=train_dataloader,
          epochs=1,
          warmup_steps=100,
          output_path='biblical_reranker_model')
print("Training Complete.")

In [ ]:
# 5. Export to ONNX
print("Exporting to ONNX...")
!optimum-cli export onnx --model biblical_reranker_model --task text-classification model_onnx/
print("Export complete. Files are in 'model_onnx/' folder.")

In [ ]:
# 6. Download Results
from google.colab import files
!zip -r biblical_reranker.zip model_onnx/
files.download('biblical_reranker.zip')